# SupplyMind v4.0-arcadia-live — Hormuz Live Demo Notebook

**Runtime**: ~5 minutes end-to-end in Google Colab (CPU tier).

This notebook:
1. Clones the SupplyMind repo.
2. Installs minimal deps.
3. Runs the live Hormuz pipeline against real 2026 news data.
4. Verifies 10 headline receipts.
5. Runs a subset of the 249+ test suite.

No GPU required. No Ollama required. No API keys required (uses cached crisis library + deterministic rubric fallback).

## 1. Clone + install

In [ ]:
# Clone the repo
!git clone --depth 1 https://github.com/ShAuRyA-Noodle/Sleep-Token.git supplymind 2>&1 | tail -5
%cd supplymind

# Install minimal deps (Colab CPU tier — no torch CUDA, no Ollama, no GPU embedders)
!pip install -q fastapi uvicorn pydantic numpy pandas networkx scipy scikit-learn matplotlib requests xgboost lightgbm catboost 2>&1 | tail -3
!pip install -q 'httpx<0.28' pytest 2>&1 | tail -2

## 2. Live Hormuz pipeline — the 90-second demo

In [ ]:
import sys
sys.path.insert(0, '.')
from versions.v4_arcadia_live.realtime.hormuz_endpoint import run_hormuz_pipeline, ScenarioRequest

req = ScenarioRequest(
    scenario_text=('Iran threatens full closure of Strait of Hormuz after US Navy seizes '
                   'Iranian cargo ship. Brent crude $123/bbl. Carriers pause Persian Gulf bookings.'),
    region='hormuz',
    enable_llm_judges=False,        # no Ollama in Colab
    include_recent_signals=False,   # no events.db
    k_analogs=3,
)
resp = run_hormuz_pipeline(req)
print('Risk level:', resp.risk_level, '\nConfidence:', resp.consensus_confidence)
print('\nTop-1 analog:', resp.analogs[0]['name'], f"(similarity {resp.analogs[0]['similarity']:.3f})")
print('\nCounterfactual:')
for k, v in resp.counterfactual.items():
    print(f'  {k}: {v:,}' if isinstance(v, (int, float)) else f'  {k}: {v}')

## 3. Recommended actions (ranked by loss-avoided / cost)

In [ ]:
for i, a in enumerate(resp.recommended_actions, 1):
    loss_avoided = (a.estimated_loss_avoided_usd or 0)
    cost = a.estimated_cost_usd or 0
    print(f'{i}. {a.action_type}')
    print(f'   reason: {a.reason[:120]}')
    if loss_avoided:
        print(f'   est. cost: ${cost:,.0f} → loss avoided: ${loss_avoided:,.0f}')
    print()

## 4. Verify headline receipts (8 of 13 — without jq)

In [ ]:
import json
receipts_to_check = [
    ('versions/v3_arcadia/results/R5_GRANITE.json', ['pipelines', 'P2_mxbai_bi', 'p1'], 'RAG P@1'),
    ('versions/v3_arcadia/results/R5_GRANITE.json', ['pipelines', 'P2_mxbai_bi', 'mrr'], 'RAG MRR'),
    ('versions/v3_arcadia/results/R5_BEIR_MANUAL.json', ['our_results', 'snowflake-arctic-l', 'mean_ndcg@10'], 'BEIR nDCG@10'),
    ('versions/v3_arcadia/results/R4_DANGEROUS_V2_ABLATION.json',
     ['agreement_primary_panel', 'krippendorff_alpha_ordinal'], 'Krippendorff α'),
    ('versions/v3_arcadia/results/R4_DANGEROUS_V2_ABLATION.json',
     ['agreement_primary_panel', 'cohen_weighted_kappa_qwen_vs_mistral'], 'Cohen κ'),
    ('versions/v3_arcadia/results/R6_GETHSEMANE_MASKING_ABLATION.json',
     ['action_masking_contribution', 'reward_pct_delta'], 'Masking lift %'),
    ('versions/v3_arcadia/results/R6_PROVIDER_V2.json',
     ['graphs', 'easy', 'improvement_vs_mlp_pct'], 'GCN easy vs MLP %'),
    ('versions/v3_arcadia/results/R3_TIMESFM_QUANTILE.json',
     ['targets', 'DCOILWTICO', 'timesfm_conf=0.95', 'dev_from_nominal'], 'TimesFM-CP dev'),
]
print(f'{'#':<4} {'claim':<24} {'value':<10}')
print('-' * 40)
for i, (f, path, name) in enumerate(receipts_to_check, 1):
    try:
        data = json.load(open(f))
        for key in path:
            data = data[key]
        print(f'{i:<4} {name:<24} {data}')
    except Exception as e:
        print(f'{i:<4} {name:<24} (failed: {e})')

## 5. Smoke test (fast subset of 249+ tests)

In [ ]:
# Run v4 fast tests (offline, no GPU needed)
!python -m pytest \
    versions/v4_arcadia_live/tests/test_spof_v2.py \
    versions/v4_arcadia_live/tests/test_stacking_v2.py \
    versions/v4_arcadia_live/tests/test_hormuz_endpoint.py \
    versions/v4_arcadia_live/tests/test_gcn_attention_viz.py \
    versions/v4_arcadia_live/tests/test_pareto_carbon.py \
    versions/v4_arcadia_live/tests/test_conformal_rl.py \
    -q --tb=line 2>&1 | tail -8

## Next steps

- **Full test suite**: `!python -m pytest tests/ versions/v4_arcadia_live/tests/ -q` (~2 min on Colab CPU).
- **Live ingestion** (needs `.env` with NEWS_API_KEY / FRED_API_KEY): 
  ```bash
  !python -m versions.v4_arcadia_live.realtime.ingestor --once --skip marinetraffic
  ```
- **Karpathy autoresearch** (needs GPU + local MaskablePPO deps):
  ```bash
  !python -m versions.v4_arcadia_live.autoresearch.orchestrator --seeds-only
  ```
- **Read the preprint**: `versions/v4_arcadia_live/docs/PREPRINT.md`
- **Judges' quick path**: `docs/v4/JUDGES.md`.